# 🔤 Old Permic OCR — Progressive Synthetic Dataset Generation

**Layer 2**: Generates synthetic training data through 12 progressive curriculum stages.

| Stage | Focus | Difficulty |
|-------|-------|------------|
| 1 | Clean isolated glyphs | Minimal |
| 2–3 | Paper textures, mild degradation | Low |
| 4–6 | Stone/parchment backgrounds, geometric variation | Medium |
| 7–9 | Multi-material, occlusions, complex scenes | High |
| 10–12 | Historical manuscript simulation | Maximum |

Each stage builds on the previous, ensuring the YOLO model is trained
progressively from simple to hard recognition tasks.

In [ ]:
# ── Cell 02: Runtime & Environment Info ──────────────────────────────────────
import subprocess, sys, platform

print(f'Python : {sys.version.split()[0]}')
print(f'Platform: {platform.system()} {platform.machine()}')

# GPU info
try:
    gpu_info = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total',
                                        '--format=csv,noheader'], text=True).strip()
    print(f'GPU     : {gpu_info}')
except Exception:
    print('GPU     : Not available (CPU mode)')

# RAM info
try:
    import psutil
    ram = psutil.virtual_memory().total / 1e9
    print(f'RAM     : {ram:.1f} GB')
except ImportError:
    print('RAM     : psutil not installed')

# Disk info
import shutil
disk = shutil.disk_usage('/content' if __import__('os').path.exists('/content') else '.')
print(f'Disk    : {disk.free / 1e9:.1f} GB free of {disk.total / 1e9:.1f} GB')

import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
# ── Cell 03: Install Package ─────────────────────────────────────────────────
import subprocess, sys

print('Installing ocroldpermic package from main branch...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'git+https://github.com/Emran025/ocroldpermic@main'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('✅ Package installed successfully.')
else:
    print('⚠️  pip install output:')
    print(result.stderr[-2000:])

# Also ensure pyyaml is available
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'], capture_output=True)

In [ ]:
# ── Cell 04: GitHub Authentication ───────────────────────────────────────────
import os, getpass, stat, tempfile

def _get_token() -> str:
    # 1. Try Colab Secrets (recommended — never stored in notebook)
    try:
        from google.colab import userdata
        t = userdata.get('GITHUB_TOKEN')
        if t:
            return t
    except Exception:
        pass
    # 2. Try environment variable
    t = os.environ.get('GITHUB_TOKEN', '')
    if t:
        return t
    # 3. Prompt securely (not echoed)
    return getpass.getpass('Enter GitHub Personal Access Token (hidden): ')

_TOKEN = _get_token()
os.environ['GITHUB_TOKEN'] = _TOKEN

# Write a GIT_ASKPASS helper that echoes the token without storing it
_askpass_script = tempfile.NamedTemporaryFile(
    mode='w', suffix='.sh', delete=False, prefix='/tmp/git_askpass_'
)
_askpass_script.write('#!/bin/sh\necho "$GITHUB_TOKEN"\n')
_askpass_script.close()
os.chmod(_askpass_script.name, stat.S_IRWXU)
os.environ['GIT_ASKPASS'] = _askpass_script.name

print('✅ GitHub token configured (never printed or stored to disk).')

In [ ]:
# ── Cell 05: Clone / Update Code Repository ───────────────────────────────────
import os, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Emran025/ocroldpermic"
CODE_BRANCH = "main"  # Active package code and font/svg
IMAGE_BRANCH = "colab-generated-images"  # Generated image datasets only
CHECKPOINT_BRANCH = "colab-checkpoints"  # Training checkpoints and model-stage artifacts
REPO_DIR = Path("/content/repo")
IMAGE_REPO_DIR = Path("/content/image-repo")
CHECKPOINT_REPO_DIR = Path("/content/checkpoint-repo")

def _run(cmd, **kwargs):
    return subprocess.run(cmd, check=True, text=True, capture_output=True, **kwargs)

def _clone_or_update(url: str, branch: str, target: Path) -> None:
    """Clone/update exactly one branch in one directory; never switch a shared clone."""
    if (target / ".git").is_dir():
        remote = _run(["git", "-C", str(target), "remote", "get-url", "origin"]).stdout.strip()
        if remote.rstrip("/") != url.rstrip("/"):
            raise RuntimeError(f"Wrong repository in {target}: {remote}")
        _run(["git", "-C", str(target), "fetch", "origin", branch])
        _run(["git", "-C", str(target), "checkout", "-B", branch, f"origin/{branch}"])
    else:
        if target.exists():
            shutil.rmtree(target)
        target.parent.mkdir(parents=True, exist_ok=True)
        _run(["git", "clone", "--branch", branch, "--depth", "1", REPO_URL, str(target)])

_run(["git", "config", "--global", "user.email", "dataset-bot@ocr-lab"])
_run(["git", "config", "--global", "user.name", "Dataset Bot"])
_clone_or_update(REPO_URL, CODE_BRANCH, REPO_DIR)
# Keep generated images in a separate checkout so generation never rewrites
# training checkpoints.
_clone_or_update(REPO_URL, IMAGE_BRANCH, IMAGE_REPO_DIR)

GLYPH_ROOT = REPO_DIR / "font" / "svg"
if not GLYPH_ROOT.is_dir():
    raise FileNotFoundError(
        f"Glyph root not found: {GLYPH_ROOT}. The glyph repository must be cloned from {CODE_BRANCH}, not {CHECKPOINT_BRANCH}."
    )
families = sorted(p.name for p in GLYPH_ROOT.iterdir() if p.is_dir())
print(f"✅ Code repository ready: {REPO_DIR} @ {CODE_BRANCH}")
print(f"✅ Font families: {families}")
print(f"✅ Image branch checkout: {IMAGE_REPO_DIR} @ {IMAGE_BRANCH}")
print(f"ℹ️  Checkpoint branch remains independent: {CHECKPOINT_REPO_DIR}")


In [ ]:
# ── Cell 06: Smoke Test Imports ───────────────────────────────────────────────
from historical_glyph_studio import GlyphStudio
from historical_glyph_curriculum import STAGES, get_stage, GenerationPlan
from historical_glyph_curriculum.github_sync.git_ops import GitManager
from historical_glyph_curriculum.resources.detection import detect_resources

print(f'✅ historical_glyph_studio imported')
print(f'✅ historical_glyph_curriculum imported — {len(STAGES)} stages defined')

for i, stage in enumerate(STAGES[:3], 1):
    print(f'   Stage {i}: {stage.name} ({len(stage.concepts)} concepts)')

## ⚙️ Configuration

Edit these values before running the generation cells.

- `GENERATION_MODE`: `'dev'` (50 samples, fast test) | `'medium'` (500) | `'full'` (stage default)
- `GLYPH_ROOT`: Path to `font/svg` directory (auto-set from repo clone above)
- `OUTPUT_ROOT`: Where to save generated datasets

In [ ]:
# ── Cell 08: Configuration ────────────────────────────────────────────────────
import os

# ── Primary settings ──────────────────────────────────────────────────────────
GENERATION_MODE = 'full'      # 'dev' (342) | 'medium' (3420) | 'full' (34200)
GLYPH_ROOT      = os.path.join(str(REPO_DIR), 'font', 'svg')
OUTPUT_ROOT     = '/content/datasets'
GITHUB_REPO     = 'https://github.com/Emran025/ocroldpermic'
IMAGE_BRANCH = 'colab-generated-images'
CHECKPOINT_BRANCH = 'colab-checkpoints'
SESSION_FILE    = '/content/generation_session.json'

# Auto-approve mode: set to True for headless / automated multi-machine execution
AUTO_APPROVE    = True

# Samples per stage per mode (covering 342 glyph variants):
# 'dev': 342 (1 per glyph)
# 'medium': 3420 (10 per glyph)
# 'full': 34200 (100 per glyph)
SAMPLES_BY_MODE = {
    'dev':    342,
    'medium': 3420,
    'full':   34200,
}
SAMPLES_OVERRIDE = SAMPLES_BY_MODE.get(GENERATION_MODE, 34200)

os.makedirs(OUTPUT_ROOT, exist_ok=True)

# Restore previously pushed stages before generation starts. The image branch is
# a Git checkout, not the active runtime directory, so resume state must be
# copied back into OUTPUT_ROOT explicitly on every fresh Colab runtime.
from shutil import copytree
_image_dataset_root = Path(IMAGE_REPO_DIR) / "datasets"
_runtime_dataset_root = Path(OUTPUT_ROOT)
if _image_dataset_root.is_dir():
    copytree(_image_dataset_root, _runtime_dataset_root, dirs_exist_ok=True)
    _restored_stages = sorted(
        p.name for p in _image_dataset_root.glob("stage_*") if p.is_dir()
    )
    print(f"Restored {len(_restored_stages)} stage dataset(s) from {IMAGE_BRANCH}: {_restored_stages}")
else:
    print(f"No prior datasets found on {IMAGE_BRANCH}; starting generation from scratch.")
print(f'Mode            : {GENERATION_MODE}')
print(f'Samples/stage   : {SAMPLES_OVERRIDE}')
print(f'Auto approve    : {AUTO_APPROVE}')
print(f'Glyph root      : {GLYPH_ROOT}')
print(f'Output root     : {OUTPUT_ROOT}')


In [ ]:
# ── Cell 09: Resource Detection & Auto-tune ───────────────────────────────────
from historical_glyph_curriculum.resources.detection import detect_resources, auto_tune

resources = detect_resources()
tuned = auto_tune(resources, GENERATION_MODE)
GENERATION_WORKERS = tuned.workers
GENERATION_BATCH_SIZE = tuned.batch_size

print(f'GPU VRAM  : {resources.gpu_vram_gb:.1f} GB')
print(f'CPU cores : {resources.cpu_count}')
print(f'RAM       : {resources.ram_gb:.1f} GB')
print()
print(f'Auto-tuned workers  : {tuned.workers}')
print(f'Auto-tuned batch_sz : {tuned.batch_size}')

In [ ]:
# ── Cell 10: Engine Setup ─────────────────────────────────────────────────────
import os
from historical_glyph_studio import GlyphStudio
from historical_glyph_curriculum.github_sync.git_ops import GitManager

# Rendering always reads from the main/code checkout.
studio = GlyphStudio(glyph_root=GLYPH_ROOT)
print(f"✅ GlyphStudio ready — {len(studio._repo.records)} glyph records indexed")

# Dataset pushes use a separate data-only checkout, never the glyph checkout.
# IMAGE_REPO_DIR was already cloned/updated in the repository setup cell.
# Reuse that checkout instead of cloning the image branch a second time.
git_manager = GitManager(
    repo_dir=IMAGE_REPO_DIR,
    remote_url=GITHUB_REPO,
    branch=IMAGE_BRANCH,
)
git_manager.configure_identity(name="Dataset Bot", email="dataset-bot@ocr-lab")
print(f"✅ GitManager ready — pushing images to {IMAGE_BRANCH} via {IMAGE_REPO_DIR}")


In [ ]:
# ── Cell 11: Approval Gate Helper ────────────────────────────────────────────
import time

def approval_gate(title: str, details: str = '', timeout: int = 30) -> bool:
    """
    Non-blocking approval gate with Jupyter event-loop integration and AUTO_APPROVE support.
    
    When AUTO_APPROVE is True, logs the decision and proceeds automatically.
    When interactive, pumps IPython kernel comm events via do_one_iteration()
    so button click events from the browser are dispatched immediately.
    """
    print(f"\n{'━'*60}")
    print(f'  {title}')
    print(f"{'━'*60}")
    if details:
        print(details)

    # Check if auto-approve is enabled in global scope
    auto = globals().get('AUTO_APPROVE', False)
    if auto:
        print('  ⚡ Auto-approved (AUTO_APPROVE = True)')
        return True

    try:
        from IPython import get_ipython
        import ipywidgets as w
        from IPython.display import display
        
        ip = get_ipython()
        decision = [None]
        btn_ok  = w.Button(description='✅ Approve & Continue', button_style='success',
                           layout=w.Layout(width='220px'))
        btn_skip = w.Button(description='⏭ Skip Stage', button_style='warning',
                            layout=w.Layout(width='160px'))
        btn_stop = w.Button(description='🛑 Stop', button_style='danger',
                            layout=w.Layout(width='120px'))
        out = w.Output()
        
        def _approve(_):
            decision[0] = True
            with out:
                print('Approved ✅')
        def _skip(_):
            decision[0] = 'skip'
            with out:
                print('Skipped ⏭')
        def _stop(_):
            decision[0] = False
            with out:
                print('Stopped 🛑')
                
        btn_ok.on_click(_approve)
        btn_skip.on_click(_skip)
        btn_stop.on_click(_stop)
        display(w.HBox([btn_ok, btn_skip, btn_stop]), out)
        
        start_time = time.time()
        while decision[0] is None:
            if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, 'do_one_iteration'):
                ip.kernel.do_one_iteration()
            time.sleep(0.05)
            if timeout and (time.time() - start_time > timeout):
                print(f'\n  ⏱ Timeout reached ({timeout}s) — auto-approving to continue.')
                return True
                
        return decision[0]
    except Exception as e:
        try:
            resp = input('Approve stage? [y/skip/n] (default y): ').strip().lower()
            if resp in ('y', 'yes', 'approve', ''):
                return True
            if resp == 'skip':
                return 'skip'
            return False
        except Exception:
            return True

print('✅ Approval gate helper ready.')


In [ ]:
# ── Cell 12: Stage Runner Function ───────────────────────────────────────────
import os, json, time
from tqdm.auto import tqdm
from pathlib import Path
from historical_glyph_curriculum import get_stage
from historical_glyph_curriculum.curriculum.plan import GenerationPlan
from historical_glyph_curriculum.parallel.executor import CurriculumExecutor
from historical_glyph_curriculum.validation.dataset import DatasetValidator
from historical_glyph_curriculum.metadata.manifest import StageManifest, save_stage_manifest
from historical_glyph_curriculum.metadata.report import print_stage_summary
from historical_glyph_curriculum.preview.grid import display_grid_in_colab, select_preview_samples

def run_stage(
    stage_id: int,
    studio,
    git_manager,
    mode: str = 'medium',
    session_file: str = SESSION_FILE,
    workers: int | None = None,
    batch_size: int | None = None,
) -> bool:
    """Generate, validate, preview, approve, and push one curriculum stage."""
    stage_out = Path(OUTPUT_ROOT) / f"stage_{stage_id:02d}"
    manifest_path = stage_out / "manifest.json"
    legacy_manifest_path = stage_out / "stage_manifest.json"
    if manifest_path.exists() or legacy_manifest_path.exists():
        print(f"\n✅ Stage {stage_id:02d} already completed (manifest exists). Skipping.")
        return True

    stage_def = get_stage(stage_id)
    print(f"\n{'═'*60}")
    print(f"  📚 Stage {stage_id:02d}: {stage_def.name}")
    print(f"  Concepts: {len(stage_def.concepts)} | Mode: {mode}")
    print(f"{'═'*60}")
    stage_out.mkdir(parents=True, exist_ok=True)

    # Build the plan required by the current executor API.
    samples_override = SAMPLES_BY_MODE.get(mode)
    total_samples = samples_override if samples_override is not None else stage_def.default_samples
    plan = GenerationPlan.build(
        stage=stage_def,
        glyph_targets=studio.available_glyph_targets(),
        total_samples=total_samples,
        output_dir=stage_out,
        global_seed=42 + stage_id,
    )
    # Resolve runtime settings here so stage execution does not depend on
    # GENERATION_WORKERS / GENERATION_BATCH_SIZE existing as notebook globals.
    if workers is None or batch_size is None:
        runtime_resources = detect_resources()
        runtime_tuned = auto_tune(runtime_resources, mode)
        workers = runtime_tuned.workers if workers is None else workers
        batch_size = runtime_tuned.batch_size if batch_size is None else batch_size
    executor = CurriculumExecutor(
        glyph_root=GLYPH_ROOT, workers=workers,
        batch_size=batch_size, backend='cpu'
    )

    def persist_concept(stage_id, concept_id, concept_output):
        # Publish every completed concept immediately. A restart can resume from
        # generation_state.json without losing already-uploaded image/label pairs.
        git_manager.push_stage(
            stage_id=stage_id, files=[], token=os.environ.get('GITHUB_TOKEN', ''),
            output_dir=str(concept_output)
        )
        print(f'  ☁️  Persisted Stage {stage_id:02d} / Concept {concept_id:02d} to {IMAGE_BRANCH}')
    state_path = stage_out / "generation_state.json"
    t0 = time.time()
    concept_progress = {"bar": None, "name": None}

    def show_concept_progress(done, total, concept_name):
        """Render a live sample counter for the concept currently generating."""
        bar = concept_progress["bar"]
        if bar is None or concept_progress["name"] != concept_name:
            if bar is not None:
                bar.close()
            concept_progress["name"] = concept_name
            concept_progress["bar"] = tqdm(
                total=total,
                desc=f"Concept: {concept_name}",
                unit="img",
                leave=True,
            )
            bar = concept_progress["bar"]
        bar.update(max(0, done - bar.n))
        if done >= total:
            bar.close()
            concept_progress["bar"] = None
            concept_progress["name"] = None

    summary = executor.generate_stage(
        plan=plan,
        state_path=state_path,
        progress_callback=show_concept_progress,
        concept_callback=persist_concept,
    )
    elapsed = time.time() - t0
    print(f"  Generated {summary['total_images']} samples in {elapsed:.1f}s")

    # Preview generated image files.
    try:
        samples = select_preview_samples(stage_out / "images", n=16)
        display_grid_in_colab(samples, title=f"Stage {stage_id:02d} Preview")
    except Exception as e:
        print(f"  Preview skipped: {e}")

    validator = DatasetValidator()
    report = validator.validate(
        stage_out / "images", stage_out / "labels",
        expected_class_count=len(studio.available_class_names())
    )
    print(report.summary())
    if not report.is_valid:
        print(f"  ❌ Validation failed: {report.issues[:3]}")
        return False

    manifest = StageManifest(
        stage_id=stage_id,
        stage_name=summary["stage_name"],
        total_images=summary["total_images"],
        class_distribution=summary["class_distribution"],
        materials_used=summary["materials_used"],
        families_used=summary["families_used"],
        styles_used=summary.get("styles_used", []),
        resolution_range=stage_def.canvas_size,
        seed=plan.global_seed,
        approved=False,
        commit_hash=None,
        generation_time_seconds=summary["generation_time_seconds"],
        dataset_contract={
            'image_format': 'png', 'label_format': 'yolo-5-normalized',
            'class_count': len(studio.available_class_names()),
            'label_fields': 5, 'dataset_root': f'datasets/stage_{stage_id:02d}',
        },
    )
    print_stage_summary(manifest)

    details = (
        f"  Samples    : {manifest.total_samples}\n"
        f"  Validation : PASS\n"
        f"  Time       : {elapsed:.1f}s"
    )
    decision = approval_gate(f"Stage {stage_id:02d} — {stage_def.name}", details)
    if decision is False:
        print("  Generation stopped by user.")
        return False
    if decision == "skip":
        print("  Stage skipped by user.")
        return True

    manifest.approved = True
    save_stage_manifest(manifest, stage_out)
    git_manager.push_stage(
        stage_id=stage_id,
        files=[str(manifest_path)],
        token=os.environ.get("GITHUB_TOKEN", ""),
        output_dir=str(stage_out),
    )
    print(f"  ✅ Stage {stage_id:02d} force-pushed to {IMAGE_BRANCH} only")
    return True

print("✅ run_stage() function ready.")


In [ ]:
# ── Stages 01–04: Foundation ──────────────────────────────────────────────────
for stage_id in range(1, 5):
    ok = run_stage(stage_id, studio, git_manager, GENERATION_MODE)
    if not ok:
        print(f'Stopping at stage {stage_id}. Re-run this cell to retry.')
        break
else:
    print('\n✅ Stages 1–4 complete!')

In [ ]:
# ── Stages 05–08: Intermediate ────────────────────────────────────────────────
for stage_id in range(5, 9):
    ok = run_stage(stage_id, studio, git_manager, GENERATION_MODE)
    if not ok:
        print(f'Stopping at stage {stage_id}. Re-run this cell to retry.')
        break
else:
    print('\n✅ Stages 5–8 complete!')

In [ ]:
# ── Stages 09–12: Advanced Historical ────────────────────────────────────────
for stage_id in range(9, 13):
    ok = run_stage(stage_id, studio, git_manager, GENERATION_MODE)
    if not ok:
        print(f'Stopping at stage {stage_id}. Re-run this cell to retry.')
        break
else:
    print('\n✅ Stages 9–12 complete!')

In [ ]:
# ── Cell 16: Master Manifest + Final Report ───────────────────────────────────
import json, os
from pathlib import Path
from historical_glyph_curriculum.metadata.manifest import build_master_manifest, save_stage_manifest

print('Building master curriculum manifest...')

stage_manifests = []
summary_rows = []

for stage_id in range(1, 13):
    stage_dir = Path(OUTPUT_ROOT) / f'stage_{stage_id:02d}'
    manifest_path = stage_dir / 'manifest.json'
    if manifest_path.exists():
        with open(manifest_path) as f:
            data = json.load(f)
        stage_manifests.append(data)
        total = data.get('total_samples', '?')
        summary_rows.append(f'  ✅ Stage {stage_id:02d}: {total} samples')
    else:
        summary_rows.append(f'  ⏳ Stage {stage_id:02d}: not generated')

master = build_master_manifest(stage_manifests)
master_path = Path(OUTPUT_ROOT) / 'curriculum_manifest.json'
master_path.write_text(json.dumps(master, indent=2, ensure_ascii=False))

print('\n═══════════════════════════════════════════════════════')
print('  OLD PERMIC OCR — DATASET GENERATION SUMMARY')
print('═══════════════════════════════════════════════════════')
for row in summary_rows:
    print(row)
print(f'\n  Master manifest: {master_path}')
total_completed = sum(1 for r in summary_rows if '✅' in r)
print(f'  Stages completed: {total_completed} / 12')
print('═══════════════════════════════════════════════════════')

if total_completed == 12:
    print('\n🎉 All 12 stages generated! Ready for adaptive_training.ipynb')
else:
    print(f'\n⚠️  {12 - total_completed} stage(s) still needed. Re-run generation cells.')

# Push master manifest
try:
    git_manager.push_stage(
        stage_id=0,
        files=[str(master_path)],
        token=os.environ.get("GITHUB_TOKEN", ""),
        output_dir=str(master_path.parent),
    )
    print('✅ Master image manifest force-pushed to image branch only.')
except Exception as e:
    print(f'⚠️  Could not push master manifest: {e}')